# 02 — Verify the RUOD / DUO detection data (ODverse33 Kaggle copy)

**Setup:** right panel → *Add Input* → search **"Underwater Domain in ODverse33"** (skycol) → Add.
Accelerator: none needed (CPU is fine). Internet: On (for the git clone).

This notebook answers: are the image/object counts what we expect, what are the splits and classes,
and were the images resized or recompressed (which matters for an image-quality project)?

In [ ]:
!test -d /kaggle/working/repo && git -C /kaggle/working/repo pull -q || git clone -q https://github.com/Banasri007/UW-detection-aware-iqa.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -e .
# Editable installs are invisible to an already-running kernel; make src importable directly.
import sys
if "/kaggle/working/repo/src" not in sys.path: sys.path.insert(0, "/kaggle/working/repo/src")
import glob, os
hits = glob.glob("/kaggle/input/**/Underwater/RUOD", recursive=True)
assert hits, "Attach the 'Underwater Domain in ODverse33' dataset first"
base = os.path.dirname(hits[0])
os.environ["RUOD_ROOT"] = f"{base}/RUOD"
os.environ["DUO_ROOT"] = f"{base}/DUO"
os.environ["RESULTS_ROOT"] = "/kaggle/working/results"
print(base)

In [ ]:
# Metadata files shipped with the copy
for ds in ("RUOD", "DUO"):
    for f in sorted(glob.glob(f"{base}/{ds}/*.*")):
        print(f"===== {f}"); print(open(f).read())

In [ ]:
!python scripts/build_manifests.py --only ruod duo

In [ ]:
import pandas as pd, numpy as np, yaml
from collections import Counter
from uwiqa.data import load_manifest

def class_names(ds):
    # classes.txt first: the uploader's DUO data.yaml is not valid YAML.
    d = f"{base}/{ds.upper()}"
    if os.path.exists(f"{d}/classes.txt"):
        return [l.strip() for l in open(f"{d}/classes.txt") if l.strip()]
    try:
        return yaml.safe_load(open(f"{d}/data.yaml")).get("names")
    except yaml.YAMLError as e:
        print(f"{ds}: data.yaml unreadable ({e.problem}); using class ids")
        return None

for ds in ("ruod", "duo"):
    m = load_manifest(ds)
    names = class_names(ds)
    print(f"\n#### {ds.upper()}: {len(m)} images, {m.n_objects.sum()} objects "
          f"(paper: RUOD 14,000 / 74,903; DUO 7,782 / 74,515)")
    print(m.groupby("split").agg(images=("image", "size"), objects=("n_objects", "sum")))
    print("images with no labels:", int((m.n_objects == 0).sum()))
    cnt = Counter(c for cs in m.classes.dropna() for c in str(cs).split())
    print("images containing each class:", {(names[int(k)] if names else k): v for k, v in sorted(cnt.items(), key=lambda x: int(x[0]))})

In [ ]:
# Resolution / compression check on a random sample — resized or heavily re-encoded copies would bias IQA.
from PIL import Image
for ds in ("ruod", "duo"):
    m = load_manifest(ds).sample(300, random_state=0)
    root = os.environ[f"{ds.upper()}_ROOT"]
    sizes, bpp, q = [], [], []
    for p in m.image:
        fp = f"{root}/{p}"
        im = Image.open(fp)
        sizes.append(im.size)
        bpp.append(os.path.getsize(fp) * 8 / (im.size[0] * im.size[1]))
        qt = getattr(im, "quantization", None)
        if qt: q.append(np.mean(qt[0]))
    s = pd.Series([f"{w}x{h}" for w, h in sizes]).value_counts()
    print(f"\n#### {ds.upper()} — top resolutions:\n{s.head(8)}")
    print(f"bits/pixel median {np.median(bpp):.2f}  |  mean luma quant-table value {np.mean(q):.1f} (IJG scale: ≈6 → q95, ≈12 → q90, ≈17 → q85, ≈29 → q75, ≈58 → q50)")

In [ ]:
# Near-duplicate check (CPU, ~15-25 min for 21.8k images; hashes are cached in results/dedup).
# RUOD and DUO share source footage, so duplicates across them or across RUOD splits would leak.
!python scripts/find_duplicates.py --datasets ruod duo --max-dist 20

In [ ]:
# Eyeball the closest and the borderline leaky pairs to confirm they are real duplicates.
import matplotlib.pyplot as plt
pr = pd.read_csv("/kaggle/working/results/dedup/pairs.csv")
leaky = pr[(pr.dataset_a != pr.dataset_b) | (pr.split_a != pr.split_b)].sort_values("dist")
print(len(leaky), "leaky pairs")
show = pd.concat([leaky.head(4), leaky.tail(4)]) if len(leaky) > 8 else leaky
if len(show):
    fig, ax = plt.subplots(len(show), 2, figsize=(8, 2.6 * len(show)), squeeze=False)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, side in enumerate("ab"):
            ds = row[f"dataset_{side}"]
            ax[r, c].imshow(Image.open(f"{os.environ[ds.upper() + '_ROOT']}/{row[f'image_{side}']}").convert("RGB"))
            ax[r, c].set_title(f"{ds}/{row[f'split_{side}']}  d={row.dist}", fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# Save the manifests so later notebooks can reuse them.
!ls -la /kaggle/working/results/manifests